# Exploring Comments
Exploring the comments I got for the relevant posts.

In [49]:
import pandas as pd
from tqdm import tqdm

tqdm.pandas()

In [18]:
cols = [
    "id",
    "post_id",
    "parent_id",
    "author",
    "body",
    "score",
    "created_utc",
]

In [ ]:
comments = pd.read_parquet(
    "data/sample/subreddit_resumes_comments.parquet",
    columns=cols
)

## Exploring Comments

In [20]:
comments.columns

Index(['id', 'post_id', 'parent_id', 'author', 'body', 'score', 'created_utc'],
      dtype='str')

In [23]:
author_not_mod = comments["author"] != "AutoModerator"

In [37]:
comments[author_not_mod]["body"].to_csv("data/sample/comment_texts.csv", index=False)

In [38]:
comments[author_not_mod]["body"]

1        The part about recovering 3+ TB of SharePoint ...
2        The Intune / AVD / $7k SharePoint recover / 60...
3        I can send you my tech recruiter’s contact inf...
4        Your content was removed for advertising/solic...
7        Through what application or website we use to ...
                               ...                        
11136    Hey bro — Redditor here. Your resume has a big...
11137    Good advice from u/throwaway_juniorcv. \n\nI'd...
11138    Thanks for the feed back! For some reason I ca...
11139        I no longer see that comment either. Strange.
11140    Tailoring to the specific role is more importa...
Name: body, Length: 8101, dtype: str

In [41]:
filtered_comments = comments[author_not_mod]

## Classify Relevant Comments

In [46]:
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

llm = ChatOllama(
    model="gemma4:e4b-mlx",  # or a faster model like "qwen2.5:3b"
    temperature=0.0,
    reasoning=False,
    num_ctx=2048,
    num_predict=16,
)


class ResumeAdviceCheck(BaseModel):
    is_resume_advice: bool = Field(
        description="True if the comment gives actionable resume advice, else False."
    )


structured_llm = llm.with_structured_output(ResumeAdviceCheck)


def is_resume_advice(comment: str) -> bool:
    prompt = f"""Does this Reddit comment give actionable resume advice (feedback on wording, formatting, sections, projects, skills, or targeting)?
Return False for mod notices, paid service solicitations, questions, general job-market talk, or vague reactions.

Comment:
\"\"\"{comment[:1500]}\"\"\""""
    result: ResumeAdviceCheck = structured_llm.invoke(
        [HumanMessage(content=prompt)]
    )
    return result.is_resume_advice

In [50]:
is_relevant_comment = filtered_comments["body"].progress_apply(is_resume_advice)

  0%|          | 0/8101 [00:00<?, ?it/s]

100%|██████████| 8101/8101 [1:47:54<00:00,  1.25it/s]  


In [51]:
filtered_comments["is_relevant"] = is_relevant_comment

In [52]:
filtered_comments.to_parquet("data/sample/filtered_comments.parquet", index=False)

In [53]:
filtered_comments["is_relevant"].value_counts()

is_relevant
True     4070
False    4031
Name: count, dtype: int64

In [55]:
relevant = filtered_comments[filtered_comments["is_relevant"] == True]

In [66]:
relevant.groupby("post_id")["id"].nunique().sort_values()

post_id
1rn6j7z     1
1rsycuz     1
1rt1fff     1
1vrelno     1
1rtej6s     1
           ..
1sha3h3    14
1vt3wka    14
1vu4nff    15
1rcab3y    15
1rhr4hg    17
Name: id, Length: 1678, dtype: int64